# Fed-batch yeast: does the model predict batches it has never seen? Leave-one-out cross-validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zuhairblr/middoe/blob/main/tests/tutorials/06_validation/tutorial.ipynb)

**Cross-validation** (`validation`).

A good fit is not proof of a good model: with enough parameters almost anything fits. The honest
test is prediction. `validation` leaves each experiment out in turn, refits the model to the others
and predicts the one left out, then reports the R² of the fits (*prediction*) and of the predictions
of the unseen runs (*validation*). A model that is right predicts nearly as well as it fits; a model
that is wrong, or over-fitted, does not.

You will compare the Monod model with maintenance (**MI**, right here) and without (**MIV**) on four
batches.

In [ ]:
# Google Colab only: install MIDDoE and fetch the model file(s); does nothing elsewhere
import sys
if 'google.colab' in sys.modules:
    import subprocess, urllib.request
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/zuhairblr/middoe.git'], check=True)
    for f in ['MI.py', 'MIV.py']:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/zuhairblr/middoe/main/tests/tutorials/06_validation/' + f, f)

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

from middoe.krnl_expera import expera
from middoe.iden_parmest import parmest
from middoe.iden_uncert import uncert
from middoe.log_utils import save_rounds
from middoe.iden_valida import validation

QUICK = False   # True: tiny budgets and fewer designs, a check that everything runs (minutes), not a result.
HERE = Path.cwd()
# Start clean: expera() ADDS sheets to an existing data.xlsx, so remove what an earlier run wrote.
for old in ['data.xlsx', 'data_plots', 'modelling', 'confidence', 'post_processing', 'estimability',
            'meshindep_plots', 'design_plots', 'iden_results.jac']:
    p = HERE / old
    shutil.rmtree(p) if p.is_dir() else p.unlink(missing_ok=True)
np.random.seed(0)                                # same noise at every run

In [ ]:
theta = {'MI':  [0.25, 0.25, 0.88, 0.09],
         'MIV': [0.15270, 0.40575, 0.47529]}
models = {
    'can_m': ['MI', 'MIV'],
    'krt': {m: 'pys' for m in theta},
    'src': {m: str(HERE / f'{m}.py') for m in theta},
    'theta': {m: [x * 1.2 for x in v] for m, v in theta.items()},
    't_l': {m: [x * 0.5 for x in v] for m, v in theta.items()},
    't_u': {m: [x * 1.5 for x in v] for m, v in theta.items()},
}
insilicos = {'tr_m': 'MI', 'theta': theta['MI'], 'errt': 'abs',
             'prels': {'1': {'u1': 0.05, 'u2': 30, 'y10': 5.5}, '2': {'u1': 0.10, 'u2': 30, 'y10': 5.5},
                       '3': {'u1': 0.15, 'u2': 20, 'y10': 2.0}, '4': {'u1': 0.08, 'u2': 10, 'y10': 3.0}}}

system = {
    'tvi': {'u1': {'stps': 5, 'cvp': 'CPF', 'const': 'rel', 'min': 0.05, 'max': 0.2, 'offl': 0.01, 'offt': 0.5},
            'u2': {'stps': 5, 'cvp': 'LPF', 'const': 'rel', 'min': 5, 'max': 35, 'offl': 1, 'offt': 0.5}},
    'tvo': {'y1': {'init': 0, 'meas': True, 'sp': 10, 'offt': 0.5, 'samp_s': 1, 'samp_f': [], 'unc': 0.2},
            'y2': {'init': 0.01, 'meas': True, 'sp': 10, 'offt': 0.5, 'samp_s': 1, 'samp_f': [], 'unc': 0.2}},
    'tii': {'y10': {'min': 1, 'max': 5.5}}, 'tio': {},
    't_s': [0, 40], 't_r': 0.16, 't_d': 0,
}
iden_opt = {'meth': 'SLSQP', 'ms': False, 'ob': 'WLS', 'maxit': 60 if QUICK else 500, 'tol': 1e-8,
            'var-cov': 'M', 'sens_m': 'central', 'eps': 1e-3,
            'c_plt': False, 'f_plt': True, 'plt_s': True, 'log': False}

### Fit both models to all four batches

In [ ]:
round_data = {}
for r in (1, 2, 3, 4):
    expera(system, models, insilicos, {}, expr=r)
unc = uncert(parmest(system, models, iden_opt), system, models, iden_opt)
save_rounds(1, unc['results'], 'four batches', round_data, models, iden_opt, unc['obs'], system)
pd.Series({m: unc['results'][m]['P'] for m in models['can_m']}, name='P-test [%]').round(1)

### Leave one batch out, four times

In [ ]:
validres = validation(system, models, iden_opt, round_data)
pd.DataFrame({m: {'R² of the fits (mean)': s['prediction_mean'], 'R² of the unseen batch (mean)': s['validation_mean'],
                  'R² of the unseen batch (sd)': s['validation_std']}
              for m, s in validres['R2_stats'].items()}).T.round(4)

Both models fit well, but only MI keeps its R² when it has to **predict** a batch it has not
seen. The gap between the two columns is the warning sign of a wrong or over-fitted model.

**Try:** remove batch 4 (only three folds); use relative noise (`'errt': 'rel'`, `unc` 0.08).